In [1]:
!nvidia-smi
!pip install -q transformers peft accelerate bitsandbytes pandas

Fri Sep 18 08:21:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for name in uploaded:
    print(name)

Saving qwen25_7b_hinglish_bench_generations.csv to qwen25_7b_hinglish_bench_generations.csv
Saving qwen25_3b_hinglish_bench_generations.csv to qwen25_3b_hinglish_bench_generations.csv

Uploaded files:
qwen25_7b_hinglish_bench_generations.csv
qwen25_3b_hinglish_bench_generations.csv


In [4]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for name in uploaded:
    print(name)

Saving phi35_mini_hinglish_bench_generations.csv to phi35_mini_hinglish_bench_generations.csv
Saving hinggpt_hinglish_bench_generations.csv to hinggpt_hinglish_bench_generations.csv

Uploaded files:
phi35_mini_hinglish_bench_generations.csv
hinggpt_hinglish_bench_generations.csv


In [7]:
import os
import pandas as pd

files_to_check = [
    "hinggpt_hinglish_bench_generations.csv",
    "phi35_mini_hinglish_bench_generations.csv",
    "qwen25_3b_hinglish_bench_generations.csv",
    "qwen25_7b_hinglish_bench_generations.csv"
]

for f in files_to_check:
    print(f"\n{f}")
    print("Exists:", os.path.exists(f))
    if os.path.exists(f):
        df = pd.read_csv(f)
        print("Rows:", len(df))
        print("Columns:", list(df.columns))


hinggpt_hinglish_bench_generations.csv
Exists: True
Rows: 34
Columns: ['index', 'category', 'prompt', 'generated_response']

phi35_mini_hinglish_bench_generations.csv
Exists: True
Rows: 34
Columns: ['id', 'category', 'category_name', 'prompt', 'expected_style', 'language', 'split', 'response']

qwen25_3b_hinglish_bench_generations.csv
Exists: True
Rows: 34
Columns: ['model', 'id', 'category', 'category_name', 'prompt', 'expected_style', 'generated_text']

qwen25_7b_hinglish_bench_generations.csv
Exists: True
Rows: 34
Columns: ['id', 'category', 'category_name', 'prompt', 'expected_style', 'language', 'split', 'response']


In [8]:
!pip install -q "transformers==5.17.0" "peft==0.21.0" accelerate bitsandbytes pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 53.9 MB/s eta 0:00:00


In [9]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Phi-3.5-mini-Instruct in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    attn_implementation="eager"
)

model.config.use_cache = False

print("\n✅ Judge model loaded!")
print("Device:", next(model.parameters()).device)

Loading tokenizer...


config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]


✅ Judge model loaded!
Device: cuda:0


In [10]:
import pandas as pd
import json
import os

FILES = {
    "HingGPT": "hinggpt_hinglish_bench_generations.csv",
    "Phi-3.5-mini": "phi35_mini_hinglish_bench_generations.csv",
    "Qwen2.5-3B": "qwen25_3b_hinglish_bench_generations.csv",
    "Qwen2.5-7B": "qwen25_7b_hinglish_bench_generations.csv"
}

RESPONSE_COLUMNS = [
    "generated_response",
    "generated_text",
    "response",
    "generation",
    "output",
    "answer"
]

CRITERIA = [
    "fluency",
    "code_mixing_naturalness",
    "hindi_grammar",
    "prompt_adherence",
    "spelling_consistency",
    "overall"
]

def get_response_column(df):
    for col in RESPONSE_COLUMNS:
        if col in df.columns:
            return col
    raise ValueError(f"No response column found: {list(df.columns)}")

judge_items = []

for model_name, file_name in FILES.items():
    df = pd.read_csv(file_name)
    response_col = get_response_column(df)

    for i, row in df.iterrows():
        prompt = str(row["prompt"])
        response = str(row[response_col])

        judge_prompt = f"""You are an expert evaluator of Hinglish text.

Evaluate the MODEL RESPONSE against the USER PROMPT.

USER PROMPT:
{prompt}

MODEL RESPONSE:
{response}

Score each criterion from 1 to 5:

1. fluency
2. code_mixing_naturalness
3. hindi_grammar
4. prompt_adherence
5. spelling_consistency
6. overall

Scoring:
1 = very poor
2 = poor
3 = acceptable
4 = good
5 = excellent

IMPORTANT:
- Judge only the response shown above.
- Do not rewrite or improve the response.
- Return ONLY ONE valid JSON object.
- Do NOT include explanations.
- Do NOT use Markdown.
- Do NOT use ```json``` fences.
- All six values MUST be integers from 1 to 5.
- Use exactly these keys:

{{"fluency": 3, "code_mixing_naturalness": 3, "hindi_grammar": 3, "prompt_adherence": 3, "spelling_consistency": 3, "overall": 3}}
"""

        judge_items.append({
            "model": model_name,
            "sample": i + 1,
            "prompt": prompt,
            "response": response,
            "judge_prompt": judge_prompt
        })

print("======================================")
print("JUDGE PROMPTS PREPARED")
print("======================================")
print("Total:", len(judge_items))
print("Expected: 136")

os.makedirs("/content/llm_judge", exist_ok=True)

with open("/content/llm_judge/judge_prompts.jsonl", "w", encoding="utf-8") as f:
    for item in judge_items:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved:")
print("/content/llm_judge/judge_prompts.jsonl")

JUDGE PROMPTS PREPARED
Total: 136
Expected: 136
Saved:
/content/llm_judge/judge_prompts.jsonl


In [11]:
import json
import re
import torch
import pandas as pd
from tqdm.auto import tqdm

INPUT_FILE = "/content/llm_judge/judge_prompts.jsonl"
OUTPUT_FILE = "/content/llm_judge/hinglish_bench_llm_judge_results.jsonl"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    items = [json.loads(line) for line in f]

criteria = [
    "fluency",
    "code_mixing_naturalness",
    "hindi_grammar",
    "prompt_adherence",
    "spelling_consistency",
    "overall"
]

def extract_valid_json(text):
    text = text.strip()

    # Remove markdown fences if present
    text = re.sub(r"```(?:json)?", "", text, flags=re.IGNORECASE)
    text = text.replace("```", "").strip()

    # First try complete output
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except:
        pass

    # Try extracting the first JSON object
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)

    if match:
        try:
            obj = json.loads(match.group(0))
            if isinstance(obj, dict):
                return obj
        except:
            pass

    return None


def validate_scores(obj):
    if not isinstance(obj, dict):
        return False

    if not all(key in obj for key in criteria):
        return False

    for key in criteria:
        value = obj[key]

        # Accept only integer scores 1-5
        if isinstance(value, bool):
            return False

        if isinstance(value, int):
            score = value
        elif isinstance(value, float) and value.is_integer():
            score = int(value)
        else:
            return False

        if score < 1 or score > 5:
            return False

    return True


results = []

print("=" * 70)
print("STARTING IMPROVED LLM-AS-JUDGE")
print("=" * 70)
print("Total responses:", len(items))
print()

for idx, item in enumerate(tqdm(items, desc="Judging"), start=1):

    inputs = tokenizer(
        item["judge_prompt"],
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    scores = extract_valid_json(raw_output)
    valid = validate_scores(scores)

    row = {
        "model": item["model"],
        "sample": item["sample"],
        "prompt": item["prompt"],
        "response": item["response"],
        "raw_judge_output": raw_output,
        "valid": valid
    }

    if valid:
        for key in criteria:
            row[key] = int(scores[key])
    else:
        for key in criteria:
            row[key] = None

    results.append(row)

    # Save after every sample
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

print()
print("=" * 70)
print("JUDGING COMPLETED")
print("=" * 70)

valid_count = sum(r["valid"] for r in results)
total_count = len(results)

print(f"Valid judgments: {valid_count}/{total_count}")
print(f"Validity: {valid_count / total_count * 100:.2f}%")
print("Saved:", OUTPUT_FILE)

STARTING IMPROVED LLM-AS-JUDGE
Total responses: 136



Judging:   0%|          | 0/136 [00:00<?, ?it/s]


JUDGING COMPLETED
Valid judgments: 17/136
Validity: 12.50%
Saved: /content/llm_judge/hinglish_bench_llm_judge_results.jsonl


In [12]:
import json

path = "/content/llm_judge/hinglish_bench_llm_judge_results.jsonl"

with open(path, "r", encoding="utf-8") as f:
    results = [json.loads(line) for line in f]

invalid = [r for r in results if not r["valid"]]

print("Invalid:", len(invalid))
print("\n" + "=" * 70)

for i, r in enumerate(invalid[:10], 1):
    print(f"\nINVALID SAMPLE {i}")
    print("Model:", r["model"])
    print("Sample:", r["sample"])
    print("RAW JUDGE OUTPUT:")
    print(repr(r["raw_judge_output"]))
    print("-" * 70)

Invalid: 119


INVALID SAMPLE 1
Model: HingGPT
Sample: 1
RAW JUDGE OUTPUT:
'Return JUST the final score as described here:\n\n{\n    "fluency": 3,\n    "code_mixing_naturalness": 3,\n0\n**Reply:** {\n    "fluency": 3,\n    "code_mixing_naturalness": 3,\n    "hindi_grammar": 3,\n    "prompt_adherence": 3,\n    "spelling_consistency": 3,\n    "overall": 3\n}'
----------------------------------------------------------------------

INVALID SAMPLE 2
Model: HingGPT
Sample: 2
RAW JUDGE OUTPUT:
"Provide your answer as described without additional information. The score reflects how well the model's response aligns with what would typically expected for someone responding naturally and appropriately given the user prompt while maintaining grammatical correctness within Indian English/Hinglish context. Despite being cheerful, it lacks complete adherence to addressing stress directly related to exams which was more fitting considering the original request; hence intermediate scores across categor

In [13]:
import json
import re
import torch

test_items = items[:5]

def extract_json(text):
    text = text.strip()

    # Remove markdown fences
    text = re.sub(r"```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```\s*", "", text)

    # Find JSON object
    match = re.search(r"\{[^{}]*\}", text, flags=re.DOTALL)

    if not match:
        return None

    try:
        return json.loads(match.group(0))
    except:
        return None


for n, item in enumerate(test_items, 1):

    print("=" * 70)
    print(f"TEST {n}/5 — {item['model']} sample {item['sample']}")

    # Make the instruction extremely short
    short_prompt = f"""Evaluate this Hinglish response.

PROMPT:
{item['prompt']}

RESPONSE:
{item['response']}

Give integer scores from 1 to 5 for:
fluency, code_mixing_naturalness, hindi_grammar, prompt_adherence, spelling_consistency, overall.

Return ONLY this JSON format:
{{"fluency":3,"code_mixing_naturalness":3,"hindi_grammar":3,"prompt_adherence":3,"spelling_consistency":3,"overall":3}}
"""

    inputs = tokenizer(
        short_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1536
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            use_cache=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]

    raw = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    parsed = extract_json(raw)

    print("\nRAW OUTPUT:")
    print(repr(raw))

    print("\nPARSED:")
    print(parsed)

TEST 1/5 — HingGPT sample 1

RAW OUTPUT:
'Remember, fluency is about how smoothly the conversation flows, code mixing refers to the blending of English and Hindi without awkward transitions, hindi grammar assesses correct use of Hindi language rules, prompt adherence'

PARSED:
None
TEST 2/5 — HingGPT sample 2

RAW OUTPUT:
'Remember, fluency refers to how smoothly the text flows; code mixing naturalness assesses the seamless integration of English and Hinglish; hindi grammar evaluates correct usage of Hindi language rules; prompt adherence'

PARSED:
None
TEST 3/5 — HingGPT sample 3

RAW OUTPUT:
'Please justify each score with at least one piece of evidence from the text. The evaluation should be based strictly on the content and quality of the provided Hinglish response without considering external factors such as the original English prompt or personal opinions not grounded'

PARSED:
None
TEST 4/5 — HingGPT sample 4

RAW OUTPUT:
'Remember, the higher the score, the better. Fluency refe

In [14]:
def extract_json(text):
    text = text.strip()

    # Remove markdown fences
    text = re.sub(r"```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```\s*", "", text)

    # Extract JSON object
    match = re.search(r"\{[^{}]*\}", text, flags=re.DOTALL)

    if match:
        try:
            return json.loads(match.group(0))
        except:
            pass

    return None


for n, item in enumerate(test_items, 1):

    print("=" * 70)
    print(f"TEST {n}/5 — {item['model']} sample {item['sample']}")

    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict Hinglish response evaluator. "
                "Return ONLY a JSON object with six integer scores from 1 to 5. "
                "Do not explain your scores."
            )
        },
        {
            "role": "user",
            "content": f"""Evaluate this response.

PROMPT:
{item['prompt']}

RESPONSE:
{item['response']}

Score:
fluency
code_mixing_naturalness
hindi_grammar
prompt_adherence
spelling_consistency
overall

Return exactly:
{{"fluency":3,"code_mixing_naturalness":3,"hindi_grammar":3,"prompt_adherence":3,"spelling_consistency":3,"overall":3}}"""
        }
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat_text,
        return_tensors="pt",
        truncation=True,
        max_length=1536
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]

    raw = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    parsed = extract_json(raw)

    print("\nRAW OUTPUT:")
    print(repr(raw))

    print("\nPARSED:")
    print(parsed)

TEST 1/5 — HingGPT sample 1

RAW OUTPUT:
'{"fluency":3, "code_mixing_naturalness":3, "hindi_grammar":3, "prompt_adherence":2, "spelling_'

PARSED:
None
TEST 2/5 — HingGPT sample 2

RAW OUTPUT:
'{"fluency":3, "code_mixing_naturalness":3, "hindi_grammar":3, "prompt_adherence":3, "spelling_'

PARSED:
None
TEST 3/5 — HingGPT sample 3

RAW OUTPUT:
'{"fluency":3,"code_mixing_naturalness":3,"hindi_grammar":3,"prompt_adherence":3,"spelling_consistency":'

PARSED:
None
TEST 4/5 — HingGPT sample 4

RAW OUTPUT:
'{"fluency":3,"code_mixing_naturalness":3,"hindi_grammar":3,"prompt_adherence":3,"spelling_consistency":'

PARSED:
None
TEST 5/5 — HingGPT sample 5

RAW OUTPUT:
'{"fluency":3,"code_mixing_naturalness":3,"hindi_grammar":3,"prompt_adherence":3,"spelling_consistency":'

PARSED:
None


In [15]:
for n, item in enumerate(test_items, 1):

    print("=" * 70)
    print(f"TEST {n}/5 — {item['model']} sample {item['sample']}")

    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict Hinglish response evaluator. "
                "Return ONLY one JSON object. "
                "No explanation. No markdown. "
                "Use integer scores from 1 to 5."
            )
        },
        {
            "role": "user",
            "content": f"""Evaluate this response.

PROMPT:
{item['prompt']}

RESPONSE:
{item['response']}

Score these six criteria:
fluency, code_mixing_naturalness, hindi_grammar,
prompt_adherence, spelling_consistency, overall.

Return exactly one JSON object with these six keys."""
        }
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat_text,
        return_tensors="pt",
        truncation=True,
        max_length=1536
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]

    raw = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    parsed = extract_json(raw)

    print("RAW:", repr(raw))
    print("PARSED:", parsed)

TEST 1/5 — HingGPT sample 1
RAW: '{\n  "fluency": 3,\n  "code_mixing_naturalness": 4,\n  "hindi_grammar": 2,\nemplty,\n"prompt_adherence": 3,\n"spelling_consistency": 2,\n"overall": 3\n}'
PARSED: None
TEST 2/5 — HingGPT sample 2
RAW: '{\n  "fluency": 3,\n  "code_mixing_naturalness": 4,\n  "hindi_grammar": 3,\n  "prompt_adherence": 4,\n  "spelling_consistency": 5,\n  "overall": 4\n}\n\n\nPlease note that the flu'
PARSED: {'fluency': 3, 'code_mixing_naturalness': 4, 'hindi_grammar': 3, 'prompt_adherence': 4, 'spelling_consistency': 5, 'overall': 4}
TEST 3/5 — HingGPT sample 3
RAW: '{\n  "fluency": 2,\n  "code_mixing_naturalness": 1,\n  "hindi_grammar": 2,\n0,\n"prompt_adherence": 1,\n"spelling_consistency": 1,\n"overall": 1\n}\n\nExplanation:\n-'
PARSED: None
TEST 4/5 — HingGPT sample 4
RAW: '{\n  "fluency": 2,\n  "code_mixing_naturalness": 1,\n  "hindi_grammar": 1,\n0,\n"prompt_adherence": 1,\n"spelling_consistency": 1,\n"overall": 1\n}\n\nExplanation:\n-'
PARSED: None
TEST 5/5 — HingGP

In [16]:
for n, item in enumerate(test_items, 1):

    print("=" * 70)
    print(f"TEST {n}/5 — {item['model']} sample {item['sample']}")

    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict evaluator. "
                "Evaluate the response from 1 to 5 on six criteria. "
                "Output ONLY six integers separated by commas. "
                "Do not write any explanation or other text."
            )
        },
        {
            "role": "user",
            "content": f"""PROMPT:
{item['prompt']}

RESPONSE:
{item['response']}

Return scores in EXACTLY this order:
fluency, code_mixing_naturalness, hindi_grammar,
prompt_adherence, spelling_consistency, overall

Example output:
3,4,3,5,4,4"""
        }
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat_text,
        return_tensors="pt",
        truncation=True,
        max_length=1536
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]

    raw = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    parts = [x.strip() for x in raw.split(",")]

    valid = (
        len(parts) == 6
        and all(x.isdigit() and 1 <= int(x) <= 5 for x in parts)
    )

    print("RAW:", repr(raw))
    print("VALID:", valid)

TEST 1/5 — HingGPT sample 1
RAW: '3,5,2,5,3,4'
VALID: True
TEST 2/5 — HingGPT sample 2
RAW: '4,5,3,5,5,5'
VALID: True
TEST 3/5 — HingGPT sample 3
RAW: '4,5,5,5,5,5'
VALID: True
TEST 4/5 — HingGPT sample 4
RAW: '5,5,5,5,5,5'
VALID: True
TEST 5/5 — HingGPT sample 5
RAW: '4,5,3,5,4,4'
VALID: True


In [18]:
all_results = []

for idx, item in enumerate(judge_items, 1):

    print(f"[{idx}/136] {item['model']} sample {item['sample']}")

    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict evaluator. "
                "Evaluate the response from 1 to 5 on six criteria. "
                "Output ONLY six integers separated by commas. "
                "Do not write any explanation or other text."
            )
        },
        {
            "role": "user",
            "content": f"""PROMPT:
{item['prompt']}

RESPONSE:
{item['response']}

Return scores in EXACTLY this order:
fluency, code_mixing_naturalness, hindi_grammar,
prompt_adherence, spelling_consistency, overall

Example:
3,4,3,5,4,4"""
        }
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat_text,
        return_tensors="pt",
        truncation=True,
        max_length=1536
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][
        inputs["input_ids"].shape[1]:
    ]

    raw = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    parts = [x.strip() for x in raw.split(",")]

    valid = (
        len(parts) == 6
        and all(
            x.isdigit() and 1 <= int(x) <= 5
            for x in parts
        )
    )

    if valid:
        scores = list(map(int, parts))
    else:
        scores = [None] * 6

    all_results.append({
        "model": item["model"],
        "sample": item["sample"],

        # FIXED: category may not exist
        "category": item.get("category", ""),

        "prompt": item["prompt"],
        "response": item["response"],

        "fluency": scores[0],
        "code_mixing_naturalness": scores[1],
        "hindi_grammar": scores[2],
        "prompt_adherence": scores[3],
        "spelling_consistency": scores[4],
        "overall": scores[5],

        "raw_judge_output": raw,
        "valid": valid
    })

print("\nDONE!")
print("Total:", len(all_results))
print("Valid:", sum(r["valid"] for r in all_results))
print("Invalid:", sum(not r["valid"] for r in all_results))

[1/136] HingGPT sample 1
[2/136] HingGPT sample 2
[3/136] HingGPT sample 3
[4/136] HingGPT sample 4
[5/136] HingGPT sample 5
[6/136] HingGPT sample 6
[7/136] HingGPT sample 7
[8/136] HingGPT sample 8
[9/136] HingGPT sample 9
[10/136] HingGPT sample 10
[11/136] HingGPT sample 11
[12/136] HingGPT sample 12
[13/136] HingGPT sample 13
[14/136] HingGPT sample 14
[15/136] HingGPT sample 15
[16/136] HingGPT sample 16
[17/136] HingGPT sample 17
[18/136] HingGPT sample 18
[19/136] HingGPT sample 19
[20/136] HingGPT sample 20
[21/136] HingGPT sample 21
[22/136] HingGPT sample 22
[23/136] HingGPT sample 23
[24/136] HingGPT sample 24
[25/136] HingGPT sample 25
[26/136] HingGPT sample 26
[27/136] HingGPT sample 27
[28/136] HingGPT sample 28
[29/136] HingGPT sample 29
[30/136] HingGPT sample 30
[31/136] HingGPT sample 31
[32/136] HingGPT sample 32
[33/136] HingGPT sample 33
[34/136] HingGPT sample 34
[35/136] Phi-3.5-mini sample 1
[36/136] Phi-3.5-mini sample 2
[37/136] Phi-3.5-mini sample 3
[38/136

In [19]:
invalid_results = [
    r for r in all_results
    if not r["valid"]
]

print("Invalid count:", len(invalid_results))

for r in invalid_results:
    print("\n" + "=" * 70)
    print("Model :", r["model"])
    print("Sample:", r["sample"])
    print("RAW   :", repr(r["raw_judge_output"]))

Invalid count: 12

Model : Phi-3.5-mini
Sample: 15
RAW   : '4,5,4,5,5,5\n\nExplanation:\n-'

Model : Phi-3.5-mini
Sample: 24
RAW   : '5,5,5,5,5,5\n\nExplanation:\n-'

Model : Qwen2.5-3B
Sample: 6
RAW   : '5,5,5,5,5,5\n\nExplanation:\n-'

Model : Qwen2.5-3B
Sample: 10
RAW   : '4,5,5,5,5,5\n\nExplanation:\n-'

Model : Qwen2.5-3B
Sample: 13
RAW   : '5,5,5,5,5,5\n\n(Note: The actual review'

Model : Qwen2.5-3B
Sample: 14
RAW   : '2,3,3,3,3,3\n\nJustification:\n- flu'

Model : Qwen2.5-3B
Sample: 16
RAW   : '5,5,5,5,5,5\n\nThe story begins with the protagon'

Model : Qwen2.5-3B
Sample: 26
RAW   : '5,5,5,5,5,5\n\nExplanation:\n-'

Model : Qwen2.5-3B
Sample: 34
RAW   : '3,2,3,5,3,3\n\nExplanation:\n-'

Model : Qwen2.5-7B
Sample: 15
RAW   : '4,5,4,5,5,5\n\nExplanation:\n-'

Model : Qwen2.5-7B
Sample: 16
RAW   : '3,2,2,5,3,3\n\nExplanation:\n-'

Model : Qwen2.5-7B
Sample: 17
RAW   : '4,3,2,5,5,4\n\nExplanation:\n-'


In [20]:
import re

for r in all_results:
    raw = r["raw_judge_output"]

    # Find the first valid six-score sequence
    match = re.search(
        r'(?<!\d)([1-5])\s*,\s*([1-5])\s*,\s*([1-5])\s*,\s*'
        r'([1-5])\s*,\s*([1-5])\s*,\s*([1-5])(?!\d)',
        raw
    )

    if match:
        scores = [int(x) for x in match.groups()]

        r["fluency"] = scores[0]
        r["code_mixing_naturalness"] = scores[1]
        r["hindi_grammar"] = scores[2]
        r["prompt_adherence"] = scores[3]
        r["spelling_consistency"] = scores[4]
        r["overall"] = scores[5]

        r["valid"] = True
        r["parsed_judge_output"] = ",".join(map(str, scores))

    else:
        r["fluency"] = None
        r["code_mixing_naturalness"] = None
        r["hindi_grammar"] = None
        r["prompt_adherence"] = None
        r["spelling_consistency"] = None
        r["overall"] = None

        r["valid"] = False
        r["parsed_judge_output"] = None


print("Total:", len(all_results))
print("Valid:", sum(r["valid"] for r in all_results))
print("Invalid:", sum(not r["valid"] for r in all_results))

Total: 136
Valid: 136
Invalid: 0


In [21]:
import pandas as pd

df = pd.DataFrame(all_results)

criteria = [
    "fluency",
    "code_mixing_naturalness",
    "hindi_grammar",
    "prompt_adherence",
    "spelling_consistency",
    "overall"
]

model_results = (
    df.groupby("model")[criteria]
      .mean()
      .round(3)
)

print(model_results)

              fluency  code_mixing_naturalness  hindi_grammar  \
model                                                           
HingGPT         2.941                    3.471          2.912   
Phi-3.5-mini    3.588                    4.235          3.118   
Qwen2.5-3B      4.235                    4.441          4.235   
Qwen2.5-7B      4.235                    4.382          4.118   

              prompt_adherence  spelling_consistency  overall  
model                                                          
HingGPT                  4.147                 3.382    3.382  
Phi-3.5-mini             4.765                 4.029    3.912  
Qwen2.5-3B               4.824                 4.588    4.382  
Qwen2.5-7B               4.971                 4.588    4.441  


In [22]:
print("\nSamples per model:")
print(df["model"].value_counts())


Samples per model:
model
HingGPT         34
Phi-3.5-mini    34
Qwen2.5-3B      34
Qwen2.5-7B      34
Name: count, dtype: int64


In [23]:
std_results = (
    df.groupby("model")[criteria]
      .std()
      .round(3)
)

print(std_results)

              fluency  code_mixing_naturalness  hindi_grammar  \
model                                                           
HingGPT         1.127                    1.161          1.083   
Phi-3.5-mini    0.892                    1.103          0.880   
Qwen2.5-3B      1.103                    1.050          1.075   
Qwen2.5-7B      0.923                    1.155          1.122   

              prompt_adherence  spelling_consistency  overall  
model                                                          
HingGPT                  0.857                 1.074    1.015  
Phi-3.5-mini             0.431                 0.969    0.753  
Qwen2.5-3B               0.521                 0.857    0.922  
Qwen2.5-7B               0.171                 0.821    0.786  


In [24]:
import pandas as pd

# Overall scores for each model
overall_df = df.pivot(
    index="sample",
    columns="model",
    values="overall"
)

print("Overall score matrix:")
display(overall_df)

print("\nDescriptive statistics:")
print(
    overall_df.describe()
    .round(3)
)

Overall score matrix:


model,HingGPT,Phi-3.5-mini,Qwen2.5-3B,Qwen2.5-7B
sample,,,,
1,4,4,5,5
2,5,5,5,5
3,5,4,5,5
4,5,4,5,5
5,5,4,5,5
6,4,3,5,5
7,3,4,5,5
8,5,5,4,5
9,3,4,5,5



Descriptive statistics:
model  HingGPT  Phi-3.5-mini  Qwen2.5-3B  Qwen2.5-7B
count   34.000        34.000      34.000      34.000
mean     3.382         3.912       4.382       4.441
std      1.015         0.753       0.922       0.786
min      2.000         2.000       2.000       3.000
25%      3.000         3.250       4.000       4.000
50%      3.000         4.000       5.000       5.000
75%      4.000         4.000       5.000       5.000
max      5.000         5.000       5.000       5.000


In [25]:
print("\nMissing values:")
print(overall_df.isna().sum())

print("\nNumber of samples:")
print(overall_df.count())


Missing values:
model
HingGPT         0
Phi-3.5-mini    0
Qwen2.5-3B      0
Qwen2.5-7B      0
dtype: int64

Number of samples:
model
HingGPT         34
Phi-3.5-mini    34
Qwen2.5-3B      34
Qwen2.5-7B      34
dtype: int64


In [26]:
for model_name in overall_df.columns:
    print(model_name, sorted(overall_df[model_name].dropna().index.tolist())[:10])

HingGPT [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Phi-3.5-mini [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Qwen2.5-3B [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Qwen2.5-7B [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [27]:
from scipy.stats import friedmanchisquare

hinggpt = overall_df["HingGPT"].values
phi = overall_df["Phi-3.5-mini"].values
qwen3b = overall_df["Qwen2.5-3B"].values
qwen7b = overall_df["Qwen2.5-7B"].values

friedman_stat, friedman_p = friedmanchisquare(
    hinggpt,
    phi,
    qwen3b,
    qwen7b
)

print("Friedman test")
print("Statistic:", round(friedman_stat, 4))
print("p-value :", round(friedman_p, 6))

Friedman test
Statistic: 34.878
p-value : 0.0


In [28]:
all_results = []

for n, item in enumerate(judge_items, 1):

    print(f"[{n}/136] {item['model']} sample {item['sample']}")

    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict evaluator. "
                "Evaluate the response from 1 to 5 on six criteria. "
                "Output ONLY six integers separated by commas. "
                "Do not write any explanation or other text."
            )
        },
        {
            "role": "user",
            "content": f"""PROMPT:
{item['prompt']}

RESPONSE:
{item['response']}

Return scores in EXACTLY this order:
fluency, code_mixing_naturalness, hindi_grammar,
prompt_adherence, spelling_consistency, overall

Example:
3,4,3,5,4,4"""
        }
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat_text,
        return_tensors="pt",
        truncation=True,
        max_length=1536
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]

    raw = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    parts = [x.strip() for x in raw.split(",")]

    if (
        len(parts) == 6
        and all(x.isdigit() and 1 <= int(x) <= 5 for x in parts)
    ):
        scores = list(map(int, parts))

        all_results.append({
            "model": item["model"],
            "sample": item["sample"],
            "fluency": scores[0],
            "code_mixing_naturalness": scores[1],
            "hindi_grammar": scores[2],
            "prompt_adherence": scores[3],
            "spelling_consistency": scores[4],
            "overall": scores[5]
        })
    else:
        print("INVALID:", repr(raw))

print("\nDONE")
print("Valid results:", len(all_results), "/ 136")

[1/136] HingGPT sample 1
[2/136] HingGPT sample 2
[3/136] HingGPT sample 3
[4/136] HingGPT sample 4
[5/136] HingGPT sample 5
[6/136] HingGPT sample 6
[7/136] HingGPT sample 7
[8/136] HingGPT sample 8
[9/136] HingGPT sample 9
[10/136] HingGPT sample 10
[11/136] HingGPT sample 11
[12/136] HingGPT sample 12
[13/136] HingGPT sample 13
[14/136] HingGPT sample 14
[15/136] HingGPT sample 15
[16/136] HingGPT sample 16
[17/136] HingGPT sample 17
[18/136] HingGPT sample 18
[19/136] HingGPT sample 19
[20/136] HingGPT sample 20
[21/136] HingGPT sample 21
[22/136] HingGPT sample 22
[23/136] HingGPT sample 23
[24/136] HingGPT sample 24
[25/136] HingGPT sample 25
[26/136] HingGPT sample 26
[27/136] HingGPT sample 27
[28/136] HingGPT sample 28
[29/136] HingGPT sample 29
[30/136] HingGPT sample 30
[31/136] HingGPT sample 31
[32/136] HingGPT sample 32
[33/136] HingGPT sample 33
[34/136] HingGPT sample 34
[35/136] Phi-3.5-mini sample 1
[36/136] Phi-3.5-mini sample 2
[37/136] Phi-3.5-mini sample 3
[38/136

In [29]:
import pandas as pd

judge_df = pd.DataFrame(all_results)

judge_df.to_csv(
    "/content/hinglish_bench_llm_judge_results.csv",
    index=False
)

judge_df.to_json(
    "/content/hinglish_bench_llm_judge_results.json",
    orient="records",
    indent=2
)

print(judge_df.shape)
print(judge_df.head())

(124, 8)
     model  sample  fluency  code_mixing_naturalness  hindi_grammar  \
0  HingGPT       1        4                        5              3   
1  HingGPT       2        4                        5              3   
2  HingGPT       3        4                        5              5   
3  HingGPT       4        5                        5              5   
4  HingGPT       5        5                        5              5   

   prompt_adherence  spelling_consistency  overall  
0                 5                     3        4  
1                 5                     5        5  
2                 5                     5        5  
3                 5                     5        5  
4                 5                     5        5  


In [7]:
import os

print(os.listdir("/content"))

['.config', 'sample_data']
